In [ ]:
%load_ext dotenv
%dotenv
%load_ext mypy_ipython

In [ ]:
from langgraph.graph import START, END, StateGraph,add_messages
from typing_extensions import TypedDict
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage, BaseMessage ,AIMessage
from collections.abc import Sequence
from typing import Literal ,Annotated

In [ ]:
chat = ChatOpenAI(
    model = 'gpt-4o',
    seed = 365,
    temperature = 0.9,
    max_tokens=500
)

In [ ]:
my_list = add_messages(
    [
        HumanMessage(content=f"Hey i'm Akhil"),
        AIMessage(content=f"Hey Akhil How Can i Help you today ?"),
    ],[
        HumanMessage(content=f"Hey summerize the News in India Today ?"),
    ]
)
print(my_list)

In [ ]:
class State(TypedDict):
    message: Annotated[Sequence[BaseMessage] , add_messages]

In [ ]:
def ask_question(state : State) -> State:
    print(f'\n---------------ask_question Started')
    print('What is your Question?')
    for i in state['message']:
        i.pretty_print()

    question = 'What is your Question?'
    print(question)

    return  State(
        message=[AIMessage(content=question),HumanMessage(content=input())]
    )

In [ ]:
ask_question(State(message=[]))

In [ ]:
def chat_bot(state : State) -> State:
    print(f'\n---------------chat_bot Started')
    response = chat.invoke(
        state['message']
    )

    for i in state['message']:
        i.pretty_print()

    response.pretty_print()
    return State(
        message=[response]
    )

In [ ]:
def ask_another_question(state: State) -> State:
    print(f"\n-------> ENTERING ask_another_question:")
    question = "Would you like to ask one more question (yes/no)?"
    for i in state['message']:
        i.pretty_print()
    print(question)

    return State(
        message = [AIMessage(content=question),HumanMessage(content=input())]
    )

In [ ]:
ask_another_question(State(message=[]))

In [ ]:
def route_function(state: State) -> Literal['ask_question', END]:
    # -1 Brings the last Value
    if state['message'][-1].content.lower() == 'yes':
        return 'ask_question'
    return END

##Build Graph

In [ ]:
graph = StateGraph(State)

In [ ]:
graph.add_node("ask_question", ask_question)
graph.add_node("chat_bot", chat_bot)
graph.add_node("ask_another_question", ask_another_question)

In [ ]:
graph.add_edge(START,'ask_question')
graph.add_edge('ask_question','chat_bot')
graph.add_edge('chat_bot','ask_another_question')
graph.add_conditional_edges(
    source='ask_another_question',
    path= route_function
)

In [ ]:
graph_compiled = graph.compile()

In [ ]:
graph_compiled.invoke(
    State(message=[])
)